---
title: How the scoring rule shaped the ranking
description: The eight countries at the bottom of the September 2026 ranking are the eight where YouTube does not offer channel memberships. This chapter checks that and compares the table rescored under the October rule.
---

Under the September rule, a likely monetized verdict needed a Join button. A Join button means the channel offers memberships, and YouTube offers channel memberships only in some locations. In a country that is not on that list, no channel can show the button, so no channel can be rated likely monetized, whatever else it shows.

This chapter checks how much of the September ranking that explains. Besides the snapshot it uses two files from `analysis/2026-10-membership-rule/data`:

- the locations where YouTube Help listed channel memberships as available on 20 October 2026, and
- the country table as MonetizationKit recomputed it on 19 October 2026, under the rule that also accepts a store or 100,000+ subscribers in place of the Join button.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import plotstyle
from plotstyle import tidy

REPO = Path("..")
EXTRA = REPO / "analysis" / "2026-10-membership-rule" / "data"


def read(path):
    # keep_default_na=False: "NA" is Namibia's country code, not a missing value
    return pd.read_csv(path, keep_default_na=False)


september = read(REPO / "youtube-monetization-signals-by-country.csv")
october = read(EXTRA / "country-monetization-recomputed-2026-10-19.csv")
memberships = read(EXTRA / "youtube-channel-memberships-locations-2026-10-20.csv")
membership_codes = set(memberships["iso2"])
# YouTube Help listed 100 locations on 20 October 2026
assert len(membership_codes) == 100


def add_shares(table):
    table = table.copy()
    table["memberships"] = table["country_code"].isin(membership_codes)
    table["likely_share"] = table["likely_monetized"] / table["channels_checked"]
    return table


september = add_shares(september)
october = add_shares(october)
print(f"September: {len(september)} countries; "
      f"October: {len(october)} countries, "
      f"{october['channels_checked'].sum():,} channels")

## The bottom eight

In [ ]:
ranked = september.sort_values("likely_share", ascending=False)
ranked = ranked.reset_index(drop=True)
without = ranked[~ranked["memberships"]]
with_memberships = ranked[ranked["memberships"]]
bottom = set(ranked.tail(len(without))["country_code"])
assert bottom == set(without["country_code"])

lowest_share = with_memberships["likely_share"].min()
lowest_with = with_memberships[with_memberships["likely_share"] == lowest_share]
print("without memberships:", ", ".join(without["country"]))
print(f"best of them {without['likely_share'].max():.1%}; "
      f"lowest with memberships: {' and '.join(lowest_with['country'])} "
      f"at {lowest_share:.1%}")

{eval}`len(without)` of the 50 countries are not on YouTube's memberships list: {eval}`", ".join(without["country"])`. They are exactly the bottom {eval}`len(without)` of the September ranking. The best of them had {eval}`f"{without['likely_share'].max():.1%}"` of its channels rated likely monetized. The lowest country that does have memberships, {eval}`" and ".join(lowest_with["country"])`, had {eval}`f"{lowest_share:.1%}"`.

In [ ]:
#| label: fig-ranking
order = september.sort_values("likely_share").reset_index(drop=True)
y = np.arange(len(order))
colours = np.where(order["memberships"], plotstyle.MEMBERSHIPS, plotstyle.NO_MEMBERSHIPS)

fig, ax = plt.subplots(figsize=(8, 12.5))
ax.hlines(y, 0, order["likely_share"], color=plotstyle.GRID, lw=1, zorder=1)
ax.scatter(order["likely_share"], y, s=42, c=colours, edgecolors=plotstyle.SURFACE, linewidths=1.5, zorder=3)
ax.set_yticks(y, order["country"], fontsize=8.5)
ax.set_ylim(-0.8, len(order) - 0.2)
ax.set_xlim(-0.012, 0.55)  # a little room so dots at 0% are not cut
ax.xaxis.set_major_formatter(plotstyle.PERCENT)
ax.xaxis.set_ticks_position("top")
ax.grid(axis="x", color=plotstyle.GRID, lw=0.8)
tidy(ax)
ax.spines["bottom"].set_visible(False)
ax.axhline(len(without) - 0.5, color=plotstyle.INK_MUTED, lw=0.8, ls=(0, (3, 3)))
ax.text(0.54, (len(without) - 1) / 2, f"No channel memberships\nin these {len(without)}", ha="right", va="center",
        fontsize=9, color=plotstyle.INK_SECONDARY)
fig.suptitle("Likely monetized share, September 2026 rule", x=0.02, ha="left", fontsize=13, fontweight="semibold")
fig.text(0.02, 0.958, "Colour: whether YouTube offers channel memberships in the country (YouTube Help, 20 October 2026).",
         fontsize=9, color=plotstyle.INK_SECONDARY)
fig.legend(
    handles=[
        plt.Line2D([], [], marker="o", ls="", color=plotstyle.MEMBERSHIPS, markersize=7, label="Memberships offered"),
        plt.Line2D([], [], marker="o", ls="", color=plotstyle.NO_MEMBERSHIPS, markersize=7, label="Not offered"),
    ],
    loc="upper left", bbox_to_anchor=(0.015, 0.948), ncol=2, frameon=False, fontsize=9,
)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

In [ ]:
counts = ["channels_checked", "likely_monetized"]
totals = september.groupby("memberships")[counts].sum()
pooled_share = totals["likely_monetized"] / totals["channels_checked"]

# How much of the variation in the likely share across countries the yes/no
# memberships split accounts for (1 minus within-group / total sum of squares)
share = september["likely_share"]
group_mean = september.groupby("memberships")["likely_share"].transform("mean")
within_groups = ((share - group_mean) ** 2).sum()
overall = ((share - share.mean()) ** 2).sum()
explained = 1 - within_groups / overall
print(f"pooled likely share: with {pooled_share[True]:.1%}, "
      f"without {pooled_share[False]:.1%}; explained {explained:.1%}")

Pooled, the {eval}`len(with_memberships)` countries with memberships had {eval}`f"{pooled_share[True]:.1%}"` of their channels rated likely monetized, the {eval}`len(without)` others {eval}`f"{pooled_share[False]:.1%}"`. That single yes-or-no fact about each country accounts for {eval}`f"{explained:.0%}"` of the variation in the likely monetized share across all 50 countries.

## Same channels, new rule

The October table re-scored every stored check under the revised rule, where a Join button, a store, or 100,000+ subscribers can each complete a likely monetized verdict. It also contains channels that people looked up on MonetizationKit after September, so not every row is the same sample. To compare like with like, the next cell keeps the countries whose channel count changed by six or fewer, so that about 90% or more of each row is the September sample scored again.

In [ ]:
both = september.merge(october, on="country_code", suffixes=("_sep", "_oct"))
both["channel_change"] = (
    both["channels_checked_oct"] - both["channels_checked_sep"]
)
same_sample = both[both["channel_change"].abs() <= 6].copy()
same_sample["rise"] = (
    same_sample["likely_share_oct"] - same_sample["likely_share_sep"]
)

by_group = same_sample.groupby("memberships_sep")[
    ["likely_share_sep", "likely_share_oct", "rise"]
].mean()
same_without = int((~same_sample["memberships_sep"]).sum())


def rank_correlation(a, b):
    """Spearman's rho: Pearson on average ranks, so no SciPy is needed."""
    return float(np.corrcoef(pd.Series(a).rank(), pd.Series(b).rank())[0, 1])


rho_same = rank_correlation(
    same_sample["likely_share_sep"], same_sample["likely_share_oct"]
)
iraq = both.set_index("country_code").loc["IQ"]
tanzania = both.set_index("country_code").loc["TZ"]
group_names = {True: "memberships offered", False: "not offered"}
by_group.rename(index=group_names).style.format("{:.1%}")

{eval}`len(same_sample)` countries pass that filter, {eval}`same_without` of them without memberships. In those {eval}`same_without`, the average likely monetized share went from {eval}`f"{by_group.loc[False, 'likely_share_sep']:.1%}"` to {eval}`f"{by_group.loc[False, 'likely_share_oct']:.1%}"`. Membership countries rose as well, from {eval}`f"{by_group.loc[True, 'likely_share_sep']:.1%}"` to {eval}`f"{by_group.loc[True, 'likely_share_oct']:.1%}"`, because stores and large channels now count everywhere. Iraq went from {eval}`int(iraq["likely_monetized_sep"])` to {eval}`int(iraq["likely_monetized_oct"])` likely monetized channels out of {eval}`int(iraq["channels_checked_oct"])`, Tanzania from {eval}`int(tanzania["likely_monetized_sep"])` to {eval}`int(tanzania["likely_monetized_oct"])` out of {eval}`int(tanzania["channels_checked_oct"])`.

In [ ]:
#| label: fig-slope
fig, ax = plt.subplots(figsize=(7.5, 7.5))
for _, row in same_sample.sort_values("memberships_sep", ascending=False).iterrows():
    offered = row["memberships_sep"]
    ax.plot(
        [0, 1],
        [row["likely_share_sep"], row["likely_share_oct"]],
        color=plotstyle.MEMBERSHIPS if offered else plotstyle.NO_MEMBERSHIPS,
        lw=1.2 if offered else 2,
        alpha=0.45 if offered else 1,
        marker="o",
        markersize=4 if offered else 6,
        markeredgecolor=plotstyle.SURFACE,
        zorder=2 if offered else 3,
    )
labelled = same_sample[~same_sample["memberships_sep"]].sort_values("likely_share_oct")
label_y = labelled["likely_share_oct"].to_numpy().copy()
for i in range(1, len(label_y)):  # nudge labels apart so they never overlap
    label_y[i] = max(label_y[i], label_y[i - 1] + 0.024)
for (_, row), y_text in zip(labelled.iterrows(), label_y):
    ax.text(1.05, y_text, f"{row['country_sep']}  {row['likely_monetized_sep']} to {row['likely_monetized_oct']} of {row['channels_checked_oct']}",
            va="center", fontsize=8.5, color=plotstyle.INK_SECONDARY)
ax.set_xticks([0, 1], ["September rule\n(Join button required)", "October rule\n(Join, store or 100k+ subscribers)"])
ax.set_xlim(-0.15, 1.8)
ax.set_ylim(-0.02, 0.85)
ax.yaxis.set_major_formatter(plotstyle.PERCENT)
ax.grid(axis="y", color=plotstyle.GRID, lw=0.8)
tidy(ax)
ax.spines["bottom"].set_visible(False)
ax.set_ylabel("Share rated likely monetized")
ax.legend(
    handles=[
        plt.Line2D([], [], color=plotstyle.MEMBERSHIPS, lw=1.5, label=f"Memberships offered ({len(same_sample) - same_without} countries)"),
        plt.Line2D([], [], color=plotstyle.NO_MEMBERSHIPS, lw=2, label=f"Not offered ({same_without} countries)"),
    ],
    loc="upper left", frameon=False, fontsize=9,
)
fig.suptitle("Same countries, new rule", x=0.02, ha="left", fontsize=13, fontweight="semibold")
fig.text(0.02, 0.925, f"{len(same_sample)} countries whose sample changed by six channels or fewer between the two tables",
         fontsize=9, color=plotstyle.INK_SECONDARY)
fig.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

In [ ]:
october_ranked = october.sort_values("likely_share", ascending=False)
october_ranked = october_ranked.reset_index(drop=True)
october_ranked["rank"] = october_ranked.index + 1
ranks_without = october_ranked.loc[~october_ranked["memberships"], "rank"]
top10_september = set(ranked.head(10)["country_code"])
top10_october = set(october_ranked.head(10)["country_code"])
top10_both = len(top10_september & top10_october)
print(f"October ranks of the eight: {ranks_without.min()} to "
      f"{ranks_without.max()} of {len(october)}; top-10 overlap {top10_both}; "
      f"rank correlation (same sample) {rho_same:.2f}")

In the full October table of {eval}`len(october)` countries, the eight countries without memberships rank between {eval}`int(ranks_without.min())` and {eval}`int(ranks_without.max())` instead of filling the last eight places. Only {eval}`top10_both` of the September top ten are still in the October top ten, and the rank correlation between the two rules, over the same-sample countries, is {eval}`f"{rho_same:.2f}"`. The order of countries in the September table owed a good deal to the scoring rule.

## What to take from this

A missing signal is only evidence where the signal could have appeared. Channel memberships, stores and the Partner Program itself are not offered in every country, so a comparison between countries that leans on them has to check availability first. The September snapshot stays on Zenodo as it was published; read its bottom eight rows with this chapter in mind. The live table on MonetizationKit uses the October rule.

The folder `analysis/2026-10-membership-rule` in the repository has a longer version of this analysis, with a simulation of how much spread sampling noise alone would produce.